In [14]:
import os
from dotenv import load_dotenv

load_dotenv()

MLFLOW_TRACKING_URI = os.getenv("MLFLOW_TRACKING_URI")
MLFLOW_TRACKING_USERNAME = os.getenv("MLFLOW_TRACKING_USERNAME")
MLFLOW_TRACKING_PASSWORD = os.getenv("MLFLOW_TRACKING_PASSWORD")

In [15]:
os.environ["MLFLOW_TRACKING_URI"]= MLFLOW_TRACKING_URI
os.environ["MLFLOW_TRACKING_USERNAME"]= MLFLOW_TRACKING_USERNAME
os.environ["MLFLOW_TRACKING_PASSWORD"]= MLFLOW_TRACKING_PASSWORD

In [3]:
%pwd

'/Users/pawanpahune/End to End Deep Learning Project /research'

In [4]:
os.chdir("../")

In [5]:
%pwd

'/Users/pawanpahune/End to End Deep Learning Project '

In [6]:
import tensorflow as tf

In [7]:
model = tf.keras.models.load_model("artifacts/training/model.h5")

In [8]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class EvaluationConfig:
    path_of_model: Path
    training_data: Path
    all_params: dict
    mlflow_uri: str
    params_image_size: list
    params_batch_size: int

In [9]:
from End_to_End_DL_Project.constants import *
from End_to_End_DL_Project.utils.common import read_yaml, create_directories, save_json
import tensorflow as tf

In [10]:
class ConfigurationManager:
    def __init__(
        self, 
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        create_directories([self.config.artifacts_root])

    def get_evaluation_config(self) -> EvaluationConfig:
        # Correctly pointing to your Data folder
        training_data_path = os.path.join(self.config.data_ingestion.unzip_dir, "Data")
        
        eval_config = EvaluationConfig(
            path_of_model="artifacts/training/model.h5", # or model.h5 depending on what you saved it as
            training_data=training_data_path,
            mlflow_uri=os.environ.get("MLFLOW_TRACKING_URI"), # Pulls securely from environment variables
            all_params=self.params,
            params_image_size=self.params.IMAGE_SIZE,
            params_batch_size=self.params.BATCH_SIZE
        )
        return eval_config




In [11]:
import tensorflow as tf
from pathlib import Path
import mlflow
import mlflow.keras
from urllib.parse import urlparse
import numpy as np
from sklearn.metrics import confusion_matrix, f1_score, precision_score, recall_score
import matplotlib.pyplot as plt
import seaborn as sns

/Users/pawanpahune/End to End Deep Learning Project /venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [12]:
class Evaluation:
    def __init__(self, config: EvaluationConfig):
        self.config = config

    def _valid_generator(self):
        # 1. Use DenseNet preprocessing instead of standard rescaling
        datagenerator_kwargs = dict(
            preprocessing_function=tf.keras.applications.densenet.preprocess_input
        )

        dataflow_kwargs = dict(
            target_size=self.config.params_image_size[:-1],
            batch_size=self.config.params_batch_size,
            interpolation="bilinear",
            class_mode="categorical" # Essential for 4 classes
        )

        valid_datagenerator = tf.keras.preprocessing.image.ImageDataGenerator(
            **datagenerator_kwargs
        )

        # 2. Point directly to your specific validation folder
        valid_dir = os.path.join(self.config.training_data, "test")

        self.valid_generator = valid_datagenerator.flow_from_directory(
            directory=valid_dir,
            shuffle=False, # MUST be False to match predictions with true labels
            **dataflow_kwargs
        )

    @staticmethod
    def load_model(path: Path) -> tf.keras.Model:
        # Load with compile=False to avoid the optimizer bug, then recompile for evaluation
        model = tf.keras.models.load_model(path, compile=False)
        model.compile(loss='categorical_crossentropy', metrics=['accuracy'])
        return model
    
    def evaluation(self):
        self.model = self.load_model(self.config.path_of_model)
        self._valid_generator()
        
        print("Evaluating basic metrics...")
        self.score = self.model.evaluate(self.valid_generator)
        
        print("Generating detailed predictions for Confusion Matrix & F1...")
        # Get raw probabilities and convert to class predictions
        y_pred_probs = self.model.predict(self.valid_generator)
        y_pred = np.argmax(y_pred_probs, axis=1)
        y_true = self.valid_generator.classes
        class_labels = list(self.valid_generator.class_indices.keys())

        # Calculate advanced metrics (weighted accounts for class imbalances)
        self.f1 = f1_score(y_true, y_pred, average='weighted')
        self.precision = precision_score(y_true, y_pred, average='weighted')
        self.recall = recall_score(y_true, y_pred, average='weighted')

        # Generate and save Confusion Matrix plot
        cm = confusion_matrix(y_true, y_pred)
        plt.figure(figsize=(10,8))
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                    xticklabels=class_labels, yticklabels=class_labels)
        plt.xlabel('Predicted Label')
        plt.ylabel('True Label')
        plt.title('Confusion Matrix - 4 Class CT Scans')
        plt.tight_layout()
        
        self.cm_plot_path = "confusion_matrix.png"
        plt.savefig(self.cm_plot_path)
        plt.close()

        self.save_score()

    def save_score(self):
        # Save all advanced metrics locally
        scores = {
            "loss": self.score[0], 
            "accuracy": self.score[1],
            "f1_score": self.f1,
            "precision": self.precision,
            "recall": self.recall
        }
        save_json(path=Path("scores.json"), data=scores)
    
    def log_into_mlflow(self):
        mlflow.set_registry_uri(self.config.mlflow_uri)
        tracking_url_type_store = urlparse(mlflow.get_tracking_uri()).scheme
        
        with mlflow.start_run():
            mlflow.log_params(self.config.all_params)
            
            # Log all numerical metrics to MLflow
            mlflow.log_metrics({
                "loss": self.score[0], 
                "accuracy": self.score[1],
                "f1_score": self.f1,
                "precision": self.precision,
                "recall": self.recall
            })
            
            # Log the Confusion Matrix image so you can view it in the MLflow UI
            mlflow.log_artifact(self.cm_plot_path, "evaluation_plots")

            if tracking_url_type_store != "file":
                # Updated Model Name to reflect our new architecture
                mlflow.keras.log_model(self.model, "model", registered_model_name="DenseNet121Model")
            else:
                mlflow.keras.log_model(self.model, "model")

In [13]:
try:
    config = ConfigurationManager()
    eval_config = config.get_evaluation_config()
    evaluation = Evaluation(eval_config)
    evaluation.evaluation()
    evaluation.log_into_mlflow()

except Exception as e:
   raise e

Found 315 images belonging to 4 classes.
Evaluating basic metrics...
20/20 ━━━━━━━━━━━━━━━━━━━━ 11s 424ms/step - accuracy: 0.6159 - loss: 0.9613
Generating detailed predictions for Confusion Matrix & F1...
20/20 ━━━━━━━━━━━━━━━━━━━━ 11s 506ms/step


2026/06/01 11:30:36 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/01 11:30:40 WARNING mlflow.keras.save: You are saving a Keras model without specifying model signature.
Registered model 'DenseNet121Model' already exists. Creating a new version of this model...
2026/06/01 11:31:02 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: DenseNet121Model, version 3
Created version '3' of model 'DenseNet121Model'.


🏃 View run unequaled-hen-384 at: https://dagshub.com/PAWANPAHUNE/End-to-End-Deep-Learning-Project-Chest-Cancer-Detection-.mlflow/#/experiments/0/runs/6ddd31a978b04cb0acc35a5776f4f36e
🧪 View experiment at: https://dagshub.com/PAWANPAHUNE/End-to-End-Deep-Learning-Project-Chest-Cancer-Detection-.mlflow/#/experiments/0
